# Sentiment Analysis: VADER vs BERT

This notebook compares two approaches to sentiment analysis:

- **VADER**: a rule-based sentiment analysis method
- **BERT**: a transformer-based machine learning model

The goal is to show how different NLP methods interpret customer feedback, especially when the language is ambiguous, mixed, or contains negation.

In [1]:
# Install the libraries needed for this notebook.
#
# vaderSentiment:
# - provides the VADER sentiment analyzer
# - VADER is a rule-based sentiment analysis tool
#
# transformers:
# - provides access to pre-trained transformer models such as BERT
# - these models are trained on large text datasets and can capture more complex language patterns
#
# pandas:
# - helps us organize results in tables

!pip install -q vaderSentiment transformers pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.1 MB/s eta 0:00:00


In [2]:
# Import the libraries used in this notebook.
#
# pandas is used for creating and displaying tables.
# SentimentIntensityAnalyzer is the VADER sentiment analyzer.
# pipeline from transformers gives us an easy interface to use a pre-trained BERT model.

import pandas as pd

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import pipeline

## Example customer reviews

We start with a small set of customer reviews.

These examples represent short business texts that could come from surveys, online reviews, or customer support feedback.

In [3]:
# This list contains example customer reviews.
#
# The reviews include:
# - clearly positive feedback
# - clearly negative feedback
# - mixed feedback
# - more neutral or factual statements
#
# Using different types of reviews helps us compare how VADER and BERT behave.

reviews = [
    "The delivery was fast and the product quality was excellent.",
    "Customer support ignored my complaint for five days.",
    "The app is easy to use, but the payment screen crashes.",
    "The price is reasonable and checkout was smooth.",
    "I received the wrong invoice and nobody explained the problem.",
    "Premium support solved my issue in less than one hour.",
    "The package arrived late and the tracking information was unclear.",
    "The product is okay, but I expected better quality.",
    "I was charged twice for the same order.",
    "The chatbot gave helpful answers about my return request."
]

# We create a pandas DataFrame to store the reviews in a table.
#
# This makes it easier to display the text and later add sentiment results
# from both models as additional columns.

df = pd.DataFrame({"review": reviews})

# Display the dataset.

df

,review
0,The delivery was fast and the product quality ...
1,Customer support ignored my complaint for five...
2,"The app is easy to use, but the payment screen..."
3,The price is reasonable and checkout was smooth.
4,I received the wrong invoice and nobody explai...
5,Premium support solved my issue in less than o...
6,The package arrived late and the tracking info...
7,"The product is okay, but I expected better qua..."
8,I was charged twice for the same order.
9,The chatbot gave helpful answers about my retu...


## VADER sentiment analysis

VADER is a rule-based model.

It uses a dictionary of words and manually designed rules to estimate whether text is positive, negative, or neutral.

In [4]:
# Create a VADER sentiment analyzer.
#
# VADER does not need training in this notebook.
# It already contains a built-in sentiment lexicon and rules.

vader = SentimentIntensityAnalyzer()

In [5]:
# This helper function converts VADER's numerical compound score into a label.
#
# VADER returns several scores:
# - positive
# - neutral
# - negative
# - compound
#
# The compound score is the most commonly used summary score.
# It ranges approximately from -1 to +1:
# - close to +1 means very positive
# - close to -1 means very negative
# - close to 0 means neutral or mixed sentiment

def vader_label(text):

    # Get VADER sentiment scores for one text.

    score = vader.polarity_scores(text)["compound"]

    # Convert the numerical score into a human-readable label.
    #
    # These thresholds are commonly used with VADER:
    # - compound >= 0.05: positive
    # - compound <= -0.05: negative
    # - otherwise: neutral

    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

In [6]:
# Apply VADER to every review in the dataset.
#
# We add two new columns:
# - vader_compound: the numerical sentiment score
# - vader_label: the interpreted sentiment category

df["vader_compound"] = df["review"].apply(
    lambda text: vader.polarity_scores(text)["compound"]
)

df["vader_label"] = df["review"].apply(vader_label)

# Display the reviews with VADER results.

df[["review", "vader_compound", "vader_label"]]

,review,vader_compound,vader_label
0,The delivery was fast and the product quality ...,0.5719,positive
1,Customer support ignored my complaint for five...,-0.2023,negative
2,"The app is easy to use, but the payment screen...",0.2382,positive
3,The price is reasonable and checkout was smooth.,0.0000,neutral
4,I received the wrong invoice and nobody explai...,-0.7003,negative
5,Premium support solved my issue in less than o...,0.5859,positive
6,The package arrived late and the tracking info...,-0.2500,negative
7,"The product is okay, but I expected better qua...",0.6486,positive
8,I was charged twice for the same order.,-0.2023,negative
9,The chatbot gave helpful answers about my retu...,0.4215,positive


## BERT sentiment analysis

Now we use a transformer-based model.

Unlike VADER, BERT-style models learn language patterns from large datasets instead of relying only on manually written rules.

In [7]:
# Load a pre-trained sentiment analysis model using the transformers pipeline.
#
# This model returns star-based sentiment labels:
# - 1 star usually means very negative
# - 5 stars usually means very positive
#
# The model also returns a confidence score showing how confident it is
# about its prediction.

bert_sentiment = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [8]:
# This helper function applies the BERT sentiment model to one text.
#
# The model returns a dictionary with:
# - label: predicted sentiment rating, such as "1 star" or "5 stars"
# - score: confidence of the prediction

def bert_analyze(text):

    # Run the model on one review.
    #
    # The pipeline returns a list of results.
    # Since we analyze one text, we take the first result with [0].

    result = bert_sentiment(text)[0]

    # Return the model output.

    return result

In [9]:
# Apply BERT sentiment analysis to every review.
#
# We first store the full model output in a temporary column.
# Each output contains both the label and the confidence score.

df["bert_result"] = df["review"].apply(bert_analyze)

# Extract the star label from the BERT result.

df["bert_label"] = df["bert_result"].apply(lambda x: x["label"])

# Extract the model confidence score from the BERT result.

df["bert_confidence"] = df["bert_result"].apply(lambda x: x["score"])

# Display both VADER and BERT results side by side.

df[[
    "review",
    "vader_label",
    "vader_compound",
    "bert_label",
    "bert_confidence"
]]

,review,vader_label,vader_compound,bert_label,bert_confidence
0,The delivery was fast and the product quality ...,positive,0.5719,5 stars,0.697050
1,Customer support ignored my complaint for five...,negative,-0.2023,1 star,0.493989
2,"The app is easy to use, but the payment screen...",positive,0.2382,3 stars,0.453161
3,The price is reasonable and checkout was smooth.,neutral,0.0000,4 stars,0.534004
4,I received the wrong invoice and nobody explai...,negative,-0.7003,1 star,0.780130
5,Premium support solved my issue in less than o...,positive,0.5859,5 stars,0.616199
6,The package arrived late and the tracking info...,negative,-0.2500,1 star,0.396406
7,"The product is okay, but I expected better qua...",positive,0.6486,3 stars,0.805566
8,I was charged twice for the same order.,negative,-0.2023,1 star,0.471394
9,The chatbot gave helpful answers about my retu...,positive,0.4215,4 stars,0.307712


## Comparing VADER and BERT

This section helps to compare the outputs of the two approaches.

The key idea is to ask whether the label makes sense from a human perspective.

In [10]:
# Create a compact comparison table.
#
# This table makes it easier to compare:
# - the original customer review
# - VADER's sentiment label
# - VADER's numerical compound score
# - BERT's star-based sentiment label
# - BERT's confidence score

comparison = df[[
    "review",
    "vader_label",
    "vader_compound",
    "bert_label",
    "bert_confidence"
]]

# Round numerical values for easier reading.

comparison.round(3)

,review,vader_label,vader_compound,bert_label,bert_confidence
0,The delivery was fast and the product quality ...,positive,0.572,5 stars,0.697
1,Customer support ignored my complaint for five...,negative,-0.202,1 star,0.494
2,"The app is easy to use, but the payment screen...",positive,0.238,3 stars,0.453
3,The price is reasonable and checkout was smooth.,neutral,0.000,4 stars,0.534
4,I received the wrong invoice and nobody explai...,negative,-0.700,1 star,0.780
5,Premium support solved my issue in less than o...,positive,0.586,5 stars,0.616
6,The package arrived late and the tracking info...,negative,-0.250,1 star,0.396
7,"The product is okay, but I expected better qua...",positive,0.649,3 stars,0.806
8,I was charged twice for the same order.,negative,-0.202,1 star,0.471
9,The chatbot gave helpful answers about my retu...,positive,0.422,4 stars,0.308


In [11]:
# Count how many reviews VADER classified as positive, negative, or neutral.
#
# This gives a simple summary of the rule-based model's output.

df["vader_label"].value_counts()

,count
vader_label,
positive,5
negative,4
neutral,1


In [12]:
# Count how many reviews BERT assigned to each star category.
#
# This gives a simple summary of the transformer model's output.

df["bert_label"].value_counts()

,count
bert_label,
1 star,4
5 stars,2
3 stars,2
4 stars,2


## Testing negation and ambiguous language

Sentiment analysis becomes more difficult when language includes:

- negation, such as "not bad"
- mixed opinions, such as "good product but slow delivery"
- ambiguous or subtle wording

This section tests how both models behave in such cases.

In [13]:
# These examples are intentionally more difficult than simple positive or negative reviews.
#
# They include:
# - negation
# - mixed sentiment
# - subtle wording
# - sentences where positive and negative signals appear together

challenging_reviews = [
    "The product is not bad.",
    "The service is not good.",
    "The delivery was late, but the product is excellent.",
    "I expected more from this product.",
    "The app is simple, but it crashes too often.",
    "I cannot say I am disappointed.",
    "This is not the worst experience I have had.",
    "The support team was helpful, although the process was slow."
]

# Store the challenging examples in a new DataFrame.

challenge_df = pd.DataFrame({"review": challenging_reviews})

challenge_df

,review
0,The product is not bad.
1,The service is not good.
2,"The delivery was late, but the product is exce..."
3,I expected more from this product.
4,"The app is simple, but it crashes too often."
5,I cannot say I am disappointed.
6,This is not the worst experience I have had.
7,"The support team was helpful, although the pro..."


In [14]:
# Apply VADER to the challenging examples.
#
# Again, we store:
# - the compound score
# - the interpreted positive/negative/neutral label

challenge_df["vader_compound"] = challenge_df["review"].apply(
    lambda text: vader.polarity_scores(text)["compound"]
)

challenge_df["vader_label"] = challenge_df["review"].apply(vader_label)

# Apply BERT to the same challenging examples.

challenge_df["bert_result"] = challenge_df["review"].apply(bert_analyze)

challenge_df["bert_label"] = challenge_df["bert_result"].apply(lambda x: x["label"])
challenge_df["bert_confidence"] = challenge_df["bert_result"].apply(lambda x: x["score"])

# Display both model outputs side by side.

challenge_df[[
    "review",
    "vader_label",
    "vader_compound",
    "bert_label",
    "bert_confidence"
]].round(3)

,review,vader_label,vader_compound,bert_label,bert_confidence
0,The product is not bad.,positive,0.431,3 stars,0.454
1,The service is not good.,negative,-0.341,1 star,0.582
2,"The delivery was late, but the product is exce...",positive,0.723,4 stars,0.441
3,I expected more from this product.,neutral,0.000,3 stars,0.455
4,"The app is simple, but it crashes too often.",neutral,0.000,3 stars,0.430
5,I cannot say I am disappointed.,negative,-0.477,2 stars,0.414
6,This is not the worst experience I have had.,positive,0.510,2 stars,0.412
7,"The support team was helpful, although the pro...",positive,0.670,3 stars,0.506


## Single-sentence experiment

This section allows to test one sentence at a time.

In [15]:
# Choose one sentence to inspect in detail.
#
# then observe how VADER and BERT respond.

sentence = "The product is not bad."

# Analyze the sentence with VADER.

vader_scores = vader.polarity_scores(sentence)
vader_sentiment = vader_label(sentence)

# Analyze the sentence with BERT.

bert_output = bert_sentiment(sentence)[0]

# Print the results in a readable format.

print("Sentence:")
print(sentence)
print()

print("VADER result:")
print(vader_scores)
print("VADER label:", vader_sentiment)
print()

print("BERT result:")
print(bert_output)

Sentence:
The product is not bad.

VADER result:
{'neg': 0.0, 'neu': 0.584, 'pos': 0.416, 'compound': 0.431}
VADER label: positive

BERT result:
{'label': '3 stars', 'score': 0.4535726308822632}


## TASK: Sentiment analysis experiment

Work in small groups.

### Step 1
Add 5 new customer reviews.

### Step 2
Run both VADER and BERT on your new examples.

### Step 3
Find at least one case where the two models disagree.

### Discussion

1. Which model seems more reasonable for your examples?
2. Which sentences were difficult to classify?
3. Did negation cause problems?
4. Did mixed sentiment cause problems?
5. Which model would you choose for a real business application and why?